# Lexicon- and Pattern-based Aspect Extraction

This notebook extracts aspect candidates from Russian restaurant reviews using sentiment lexicons, POS-based syntactic patterns, propagation, support filtering, and contextual aspect markers.


## 1. Setup and data preprocessing


In [ ]:
!pip install pymorphy2
!pip install -U pymorphy2 -dicts -ru
!python -m spacy download ru_core_news_md
from IPython .display import display
import xml .etree .ElementTree as ET
from pandas import DataFrame
import pandas as pd
import numpy as np
import re
import string
import spacy
from collections import Counter
import pymorphy2
morph =pymorphy2 .MorphAnalyzer ()
nlp =spacy .load ('ru_core_news_md')

import collections
import operator


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
tree = ET.parse('/content/drive/MyDrive/RU_REST_SB1_TEST.xml.B')
root_test = tree.getroot()
test = []
for tt in root_test.iter('text'):
    test.append(tt.text)
tree = ET.parse('/content/drive/MyDrive/Colab Notebooks/se16_ru_rest_train.xml')
root_train = tree.getroot()
train = []
for tt in root_train.iter('text'):
    train.append(tt.text)
tree = ET.parse('/content/drive/MyDrive/SentiRuEval_rest_train.xml')
root_rest = tree.getroot()
rest = []
for tt in root_rest.iter('text'):
    rest.append(tt.text)


In [ ]:
def remove_punctuation(text):
    translator = str.maketrans(string.punctuation, ' ' * len(string.punctuation))
    text_without_punctuation = text.translate(translator)
    return text_without_punctuation

def remove_punctuation_except_dash(text):
    punctuation_to_remove = set((char for char in string.punctuation if char != '-'))
    processed_text = ''.join((char if char not in punctuation_to_remove else ' ' for char in text))
    return processed_text
text = 'Hello, World! This is an example text with punctuation - and more.'
processed_text = remove_punctuation_except_dash(text)
print(processed_text)


In [ ]:
tree = ET.parse('/content/drive/MyDrive/RU_REST_SB1_TEST.xml.B')
root = tree.getroot()
data = []
test_words = []
test_tokens_and_labels = []
list_without_puct_test = []
test_sentences = []
for sentence in root.iter('sentence'):
    text = sentence.find('text').text.strip()
    list_without_puct = remove_punctuation_except_dash(text)
    list_without_puct = list_without_puct.strip()
    list_without_puct_test.append(list_without_puct)
    test_sentences = nlp(list_without_puct)
    tokens = []
    test_tl = []
    sentence = []
    for token in test_sentences:
        if token.lemma_ != 'null' and token.pos_ != 'PUNCT' and (token.lemma_ != ' ') and (token.lemma_ != '-'):
            b = [token.lemma_, token.pos_]
            a = token.lemma_
            tokens.append(a)
            test_tl.append(b)
            sentence.append(a)
            test_words.append(a)
    test_tokens_and_labels.append(test_tl)
    test_sentences.append(sentence)


In [ ]:
tree = ET.parse('/content/drive/MyDrive/Colab Notebooks/se16_ru_rest_train.xml')
root = tree.getroot()
data = []
train_words = []
train_tokens_and_labels = []
list_without_puct_train = []
train_sentences = []
for sentence in root.iter('sentence'):
    text = sentence.find('text').text.strip()
    list_without_puct = remove_punctuation_except_dash(text)
    list_without_puct = list_without_puct.strip()
    list_without_puct_train.append(list_without_puct)
    train_sentences = nlp(list_without_puct)
    tokens = []
    train_tl = []
    sentence = []
    for token in train_sentences:
        if token.lemma_ != 'null' and token.pos_ != 'PUNCT' and (token.lemma_ != ' '):
            b = [token.lemma_, token.pos_]
            a = token.lemma_
            tokens.append(a)
            train_tl.append(b)
            sentence.append(a)
    train_sentences.append(sentence)
    train_tokens_and_labels.append(train_tl)


In [ ]:
rest_words = []
large_train_tokens_and_labels = []
list_without_puct_rest = []
large_train_sentences = []
for sentence in rest:
    list_without_puct = remove_punctuation(sentence)
    list_without_puct = list_without_puct.strip()
    list_without_puct_rest.append(list_without_puct)
    rest_sentences = nlp(list_without_puct)
    tokens = []
    rest_tl = []
    sentence = []
    for token in rest_sentences:
        if token.lemma_ != 'null' and token.pos_ != 'PUNCT' and (token.lemma_ != ' '):
            b = [token.lemma_, token.pos_]
            a = token.lemma_
            tokens.append(a)
            rest_tl.append(b)
            sentence.append(a)
            rest_words.append(a)
    large_train_tokens_and_labels.append(rest_tl)
    large_train_sentences.append(sentence)


In [ ]:
tree = ET.parse('/content/drive/MyDrive/RU_REST_SB1_TEST.xml.B')
root_test = tree.getroot()
test = []
for tt in root_test.iter('text'):
    test.append(tt.text)
opinions = root_test.findall('**/**/Opinion')
categories = [opinion.attrib['category'] for opinion in opinions]
polarities = [opinion.attrib['polarity'] for opinion in opinions]
target = [opinion.attrib['target'] for opinion in opinions]
print('# Opinions  : ', len(opinions))
target_o = []
for i in target:
    if i != 'NULL':
        i = i.translate(str.maketrans('', '', string.punctuation))
        target_o.append(i)
target_opinions = []
for i in target_o:
    doc = nlp(i)
    if len(doc) == 1:
        for token in doc:
            target_opinions.append(token.lemma_)
    else:
        m = []
        for token in doc:
            m.append(token.lemma_)
        target_opinions.append(' '.join(m))


## 2. Sentiment lexicons


In [ ]:
with open('/content/drive/MyDrive/Colab Notebooks/rusentilex_2017.txt', 'r', encoding='utf-8') as f:
    contents = f.readlines()


In [ ]:
rusentilex_adjectives = []
tonality = []
for line in contents:
    line = line.translate(str.maketrans('', '', string.punctuation))
    line = line.split(' ')
    for i in line:
        if i == 'Adj':
            rusentilex_adjectives.append(line[2])


In [ ]:
with open('/content/drive/MyDrive/ProductSentiRus.txt', 'r', encoding='utf-8') as f:
    contents = f.readlines()


In [ ]:
productsentirus_adjectives = []
tonality = []
for line in contents:
    line = line.split('\t')
    if line[0][-2:] == 'ЫЙ' or line[0][-2:] == 'ИЙ':
        productsentirus_adjectives.append(line[0].lower())


In [ ]:
with open('/content/drive/MyDrive/Kotelnikov.txt', 'r', encoding='utf-8') as f:
    contents = f.readlines()


In [ ]:
kotelnikov_adjectives = []
tonality = []
for line in contents:
    line = line.split('\t')
    if line[0][-2:] == 'ый' or line[0][-2:] == 'ий':
        kotelnikov_adjectives.append(line[0].lower())


In [ ]:
with open('/content/drive/MyDrive/BlinovNegative.txt', 'r', encoding='utf-8') as f:
    contents2 = f.readlines()


In [ ]:
with open('/content/drive/MyDrive/BlinovPositive.txt', 'r', encoding='utf-8') as f:
    contents1 = f.readlines()


In [ ]:
contents = contents1 + contents2


In [ ]:
blinov_adjectives = []
tonality = []
for line in contents:
    line = line.split('\n')
    if line[0][-2:] == 'ый' or line[0][-2:] == 'ий':
        blinov_adjectives.append(line[0].lower())


In [ ]:
data = pd.read_csv('/content/drive/MyDrive/emolex.txt', sep='\t')


In [ ]:
contents = data['Russian Word'].tolist()


In [ ]:
emolex_adjectives = []
tonality = []
for line in contents:
    line = line.split('\n')
    if line[0][-2:] == 'ый' or line[0][-2:] == 'ий':
        emolex_adjectives.append(line[0].lower())


## 3. Preprocessing and evaluation utilities



In [ ]:
def lemmatize_words(words):
    """Lemmatize a sequence of Russian words with spaCy and return unique lemmas."""
    lemmas = []
    for input_word in words:
        doc = nlp(input_word)
        for token in doc:
            lemmas.append(token.lemma_)
    return set(lemmas)

def evaluate_aspects(predicted_aspects, gold_aspects):
    """Compute set-based precision, recall, and F1 for extracted aspect terms."""
    predicted = set(predicted_aspects)
    gold = set(gold_aspects)
    true_positives = len(predicted & gold)
    precision = true_positives / len(predicted) if predicted else 0.0
    recall = true_positives / len(gold) if gold else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return (precision, recall, f1)


## 4. Syntactic pattern extraction




In [ ]:
def extract_by_syntactic_patterns(
    tokenized_sentences,
    sentiment_adjectives,
):
    """Extract aspect candidates with local POS-based syntactic rules."""
    collocations = []
    aspects = []

    for sentence in tokenized_sentences:
        for index, token_info in enumerate(sentence):
            if (
                token_info[1] == "ADJ"
                and token_info[0] in sentiment_adjectives
            ):
                sentiment_adjective = token_info[0]

                if index + 2 < len(sentence):
                    if (
                        sentence[index + 1][1] == "ADJ"
                        and sentence[index + 2][1] == "NOUN"
                    ):
                        second_adjective = sentence[index + 1][0]
                        if second_adjective not in sentiment_adjectives:
                            candidate_noun = sentence[index + 2][0]
                            collocations.append(
                                [
                                    second_adjective,
                                    candidate_noun,
                                ]
                            )
                            aspects.append(candidate_noun)

                if index + 1 < len(sentence):
                    if sentence[index + 1][1] == "NOUN":
                        candidate_noun = sentence[index + 1][0]
                        collocations.append(
                            [
                                sentiment_adjective,
                                candidate_noun,
                            ]
                        )
                        aspects.append(candidate_noun)

                if index + 4 < len(sentence):
                    if (
                        sentence[index + 1][1] == "CCONJ"
                        and sentence[index + 2][1] == "ADJ"
                        and sentence[index + 3][1] == "NOUN"
                    ):
                        second_adjective = sentence[index + 2][0]
                        if second_adjective not in sentiment_adjectives:
                            candidate_noun = sentence[index + 3][0]
                            collocations.append(
                                [
                                    sentiment_adjective,
                                    second_adjective,
                                    candidate_noun,
                                ]
                            )
                            aspects.append(candidate_noun)

                # The implementation intentionally did not guard index-1/index-2
                # here; the condition is used exactly.
                if (
                    sentence[index - 1][1] == "AUX"
                    and sentence[index - 2][1] == "NOUN"
                ):
                    candidate_noun = sentence[index - 2][0]
                    collocations.append(
                        [
                            sentiment_adjective,
                            candidate_noun,
                        ]
                    )
                    aspects.append(candidate_noun)

                if index != 0:
                    if sentence[index - 1][1] == "NOUN":
                        candidate_noun = sentence[index - 1][0]
                        collocations.append(
                            [
                                sentiment_adjective,
                                candidate_noun,
                            ]
                        )
                        aspects.append(candidate_noun)

                if index + 4 < len(sentence):
                    if (
                        sentence[index + 1][1] == "NOUN"
                        and sentence[index + 2][1] == "ADP"
                        and sentence[index + 3][1] == "NOUN"
                    ):
                        first_noun = sentence[index + 1][0]
                        preposition = sentence[index + 2][0]
                        second_noun = sentence[index + 3][0]
                        multiword_aspect = (
                            f"{first_noun} "
                            f"{preposition} "
                            f"{second_noun}"
                        )
                        collocations.append(
                            [
                                sentiment_adjective,
                                multiword_aspect,
                            ]
                        )
                        aspects.append(multiword_aspect)

                if index not in {0, 1, 2, 3}:
                    if (
                        sentence[index - 1][1] == "CCONJ"
                        and sentence[index - 2][1] == "ADJ"
                        and sentence[index - 3][1] == "AUX"
                        and sentence[index - 4][1] == "NOUN"
                    ):
                        second_adjective = sentence[index - 2][0]
                        candidate_noun = sentence[index - 4][0]

                        if second_adjective not in sentiment_adjectives:
                            collocations.append(
                                [
                                    sentiment_adjective,
                                    second_adjective,
                                    candidate_noun,
                                ]
                            )
                            aspects.append(candidate_noun)

    return collocations, aspects


In [ ]:
LEXICONS = {
    "RuSentiLex": rusentilex_adjectives,
    "ProductSentiRus": productsentirus_adjectives,
    "Kotelnikov": kotelnikov_adjectives,
    "Blinov": blinov_adjectives,
    "EmoLex": emolex_adjectives,
}

DATASETS = {
    "train": train_tokens_and_labels,
    "test": test_tokens_and_labels,
    "large_train": large_train_tokens_and_labels,
}

def run_lexicon_experiments(lexicons, datasets):
    """Run the same syntactic extraction procedure for every lexicon and dataset."""
    results = {}
    for lexicon_name, sentiment_adjectives in lexicons.items():
        results[lexicon_name] = {}
        for dataset_name, tokenized_sentences in datasets.items():
            collocations, aspects = extract_by_syntactic_patterns(tokenized_sentences, sentiment_adjectives)
            results[lexicon_name][dataset_name] = {'collocations': collocations, 'aspects': aspects}
    return results
lexicon_results = run_lexicon_experiments(LEXICONS, DATASETS)


## 5. Aspect–opinion propagation and preliminary filtering

Candidate aspects and opinion words are retained when they occur with more than one counterpart.


In [ ]:
def filter_supported_aspects_and_opinions(collocations):
    """Keep aspects and opinion words associated with more than one counterpart."""
    noun_to_adjectives = {}
    for collocation in collocations:
        noun = collocation[-1]
        adjectives_for_noun = collocation[:-1]
        noun_to_adjectives.setdefault(noun, []).extend(adjectives_for_noun)
    aspects = []
    for noun, adjectives_for_noun in noun_to_adjectives.items():
        noun_to_adjectives[noun] = set(adjectives_for_noun)
        if len(adjectives_for_noun) > 1:
            aspects.append(noun)
    adjective_to_nouns = {}
    for collocation in collocations:
        adjective = collocation[0]
        noun = collocation[-1]
        adjective_to_nouns.setdefault(adjective, []).append(noun)
    opinions = []
    for adjective, nouns_for_adjective in adjective_to_nouns.items():
        adjective_to_nouns[adjective] = set(nouns_for_adjective)
        if len(nouns_for_adjective) > 1:
            opinions.append(adjective)
    return (aspects, opinions)


In [ ]:
def apply_support_filter(experiment_results):
    """Apply the original support filter to every lexicon/dataset result."""
    filtered_results = {}
    for lexicon_name, dataset_results in experiment_results.items():
        filtered_results[lexicon_name] = {}
        for dataset_name, result in dataset_results.items():
            aspects, opinions = filter_supported_aspects_and_opinions(result['collocations'])
            filtered_results[lexicon_name][dataset_name] = {'aspects': aspects, 'opinions': opinions}
    return filtered_results
filtered_lexicon_results = apply_support_filter(lexicon_results)


## 5. Aspect–opinion propagation and preliminary filtering

The support-filtered aspect and opinion candidates are combined across the annotated training and test sets. Opinion words not already present in RuSentiLex are then added to the extraction lexicon, reproducing the original propagation step without the old intermediate variable chain.


In [ ]:
train_supported = filtered_lexicon_results['RuSentiLex']['train']
test_supported = filtered_lexicon_results['RuSentiLex']['test']
large_train_supported = filtered_lexicon_results['RuSentiLex']['large_train']
propagated_opinion_words = train_supported['opinions'] + test_supported['opinions']
additional_opinion_words = [
    opinion_word
    for opinion_word in propagated_opinion_words
    if opinion_word not in rusentilex_adjectives
]
expanded_opinion_lexicon = rusentilex_adjectives + additional_opinion_words


In [ ]:
propagated_extraction_results = {}
for dataset_name, tokenized_sentences in DATASETS.items():
    sentiment_words = (
        expanded_opinion_lexicon
        if dataset_name != "large_train"
        else propagated_opinion_words + additional_opinion_words
    )
    collocations, aspects = extract_by_syntactic_patterns(tokenized_sentences, sentiment_words)
    propagated_extraction_results[dataset_name] = {'collocations': collocations, 'aspects': aspects}


In [ ]:
supported_propagated_aspects = {}
for dataset_name, result in propagated_extraction_results.items():
    supported_aspects = filtered_lexicon_results['RuSentiLex'][dataset_name]['aspects']
    supported_propagated_aspects[dataset_name] = [
        aspect
        for aspect in result["aspects"]
        if aspect in supported_aspects
    ]


In [ ]:
russian_stopwords = stopwords.words('russian')
extra_stopwords = ['это', '-', ' ', '  ', '   ', '    ', '     ', '       ', '        ', '         ', '          ', '          ']
aspect_stopwords = russian_stopwords + extra_stopwords
clean_propagated_aspects = {
    dataset_name: [
        aspect
        for aspect in result["aspects"]
        if aspect not in aspect_stopwords
    ]
    for dataset_name, result in propagated_extraction_results.items()
}


In [ ]:
test_gold_lemmas = lemmatize_words(test_aspects)
test_metrics = evaluate_aspects(clean_propagated_aspects['test'], test_gold_lemmas)
test_metrics


## 6. Potential aspect markers

Potential markers are identified from words occurring immediately before or after aspect candidates in the large training corpus. 

In [ ]:
def filter_sentence_tokens(sentences, stopwords):
    """Remove stopwords and non-alphabetic tokens from tokenized sentences."""
    return [
        [
            token
            for token in sentence
            if token not in stopwords and token.isalpha()
        ]
        for sentence in sentences
    ]
filtered_large_train_sentences = filter_sentence_tokens(large_train_sentences, aspect_stopwords)
large_train_candidate_aspects = [
    aspect.lower()
    for aspect in clean_propagated_aspects["large_train"]
    if aspect != "-"
]

shared_test_large_train_aspects = list(
    (
        Counter(test_gold_lemmas)
        & Counter(large_train_candidate_aspects)
    ).elements()
)


In [ ]:
def count_words(words):
    """Return word frequencies sorted in descending order."""
    word_counts = collections.Counter(words)
    return dict(
        sorted(
            word_counts.items(),
            key=operator.itemgetter(1),
            reverse=True,
        )
    )

def collect_neighbor_words(sentences, target_words, direction):
    """Collect words immediately before or after target aspect candidates."""
    neighbors = []
    for sentence in sentences:
        for index, token in enumerate(sentence):
            if token not in target_words:
                continue
            if direction == 'before' and index > 0:
                neighbors.append(sentence[index - 1])
            elif direction == 'after' and index < len(sentence) - 1:
                neighbors.append(sentence[index + 1])
    return neighbors


In [ ]:
large_train_word_counts = Counter(
    token
    for sentence in large_train_sentences
    for token in sentence
)

preceding_neighbor_counts = count_words(
    collect_neighbor_words(
        large_train_sentences,
        shared_test_large_train_aspects,
        direction="before",
    )
)

following_neighbor_counts = count_words(
    collect_neighbor_words(
        large_train_sentences,
        shared_test_large_train_aspects,
        direction="after",
    )
)


In [ ]:
def score_potential_markers(
    neighbor_counts,
    corpus_word_counts,
    minimum_neighbor_count=50,
    require_verb=False,
):
    """Score potential markers using the original neighbor-frequency ratio."""
    marker_scores = {}
    for marker, neighbor_count in neighbor_counts.items():
        if marker not in corpus_word_counts or neighbor_count <= minimum_neighbor_count:
            continue
        if require_verb:
            marker_doc = nlp(marker)
            if not any((token.pos_ == 'VERB' for token in marker_doc)):
                continue
        conditional_frequency = neighbor_count / corpus_word_counts[marker]
        marker_scores[marker] = [conditional_frequency, neighbor_count]
    return sorted(
        marker_scores.items(),
        key=lambda item: item[1],
        reverse=True,
    )
preceding_marker_scores = score_potential_markers(
    preceding_neighbor_counts,
    large_train_word_counts,
    require_verb=True,
)
following_marker_scores = score_potential_markers(following_neighbor_counts, large_train_word_counts)


In [ ]:
following_aspect_markers = [
    "просто",
    "понравиться",
    "оказаться",
    "всегда",
    "состоять",
    "пасть",
    "составить",
    "заслуживать",
    "оставлять",
    "впечатлять",
    "располагать",
    "который",
    "которые",
    "которая",
]

preceding_aspect_markers = [
    "понравиться",
    "создаваться",
    "касаться",
    "покинуть",
    "поменять",
    "выбрать",
    "поблагодарить",
    "позвать",
    "заказать",
    "принести",
    "попросить",
    "оплатить",
    "выбрать",
    "приносить",
    "запомниться",
]


In [ ]:
def extract_marker_based_aspects(sentences, markers, direction):
    """Extract noun aspect candidates adjacent to potential aspect markers.

    Parameters
    ----------
    sentences : iterable
        Tokenized sentences.
    markers : collection
        Potential aspect markers identified from corpus context.
    direction : {"after", "before"}
        Whether the candidate is taken after or before the marker.
    """
    filtered_sentences = []
    for sentence in sentences:
        filtered_sentence = [word for word in sentence if word not in aspect_stopwords and word.isalpha()]
        filtered_sentences.append(filtered_sentence)
    marker_candidates = []
    for sentence in filtered_sentences:
        for index, word in enumerate(sentence):
            if word not in markers:
                continue
            if direction == 'after' and index < len(sentence) - 1:
                marker_candidates.append([word, sentence[index + 1]])
            elif direction == 'before' and index > 0:
                marker_candidates.append([word, sentence[index - 1]])
    extracted_pairs = []
    noun_aspects = []
    for marker, candidate in marker_candidates:
        doc = nlp(candidate)
        for token in doc:
            if token.pos_ == 'NOUN':
                noun_aspects.append(token.lemma_)
                extracted_pairs.append([marker, token.lemma_])
    return (extracted_pairs, noun_aspects)


In [ ]:
def run_marker_extraction(sentences):
    """Run aspect-marker extraction in both directions."""
    after_pairs, after_aspects = extract_marker_based_aspects(
        sentences,
        preceding_aspect_markers,
        direction="after",
    )
    before_pairs, before_aspects = extract_marker_based_aspects(
        sentences,
        following_aspect_markers,
        direction="before",
    )

    return {
        "after_marker": {
            "pairs": after_pairs,
            "aspects": after_aspects,
        },
        "before_marker": {
            "pairs": before_pairs,
            "aspects": before_aspects,
        },
    }


marker_results = {
    "train": run_marker_extraction(train_sentences),
    "test": run_marker_extraction(test_sentences),
    "large_train": run_marker_extraction(large_train_sentences),
}
